# درس 5: برآورد، خطای استاندارد و Bootstrap

متن کامل درس: [https://quantacademy.ir/courses/quant-math/estimation-standard-error-bootstrap/](https://quantacademy.ir/courses/quant-math/estimation-standard-error-bootstrap/)

داده‌های این دوره **شبیه‌سازی‌شده** هستند، نه قیمت واقعی بازار؛ هدف یاد دادن روش است. آموزشی است و توصیه سرمایه‌گذاری نیست.

سلول‌ها را از بالا به پایین اجرا کن (Shift+Enter).

## ۱. تخمین‌زن و تخمین

In [ ]:
import numpy as np
import pandas as pd

URL = "https://raw.githubusercontent.com/quantacademyir/quant-math/main/data/btc_daily_5y.csv"
close = pd.read_csv(URL, parse_dates=["date"], index_col="date")["close"]
r = np.log(close).diff().dropna()
n = len(r)
print(n, round(r.mean(), 5), round(r.std(), 4))

## ۲. تخمین‌زن بی‌طرف

In [ ]:
rng = np.random.default_rng(5)
samples = rng.normal(0, 0.03, size=(100_000, 5))
print(round(samples.var(axis=1, ddof=0).mean() / 0.03**2, 3), round(samples.var(axis=1, ddof=1).mean() / 0.03**2, 3))

## ۳. خطای استاندارد میانگین

In [ ]:
mean, s = r.mean(), r.std(ddof=1)
se = s / np.sqrt(n)
print(round(se, 5), round(mean / se, 2))
print(round(mean - 1.96 * se, 5), round(mean + 1.96 * se, 5))

In [ ]:
rng = np.random.default_rng(6)
sigma = 0.03
means = rng.normal(0, sigma, size=(20_000, n)).mean(axis=1)
print(round(means.std(), 5), round(sigma / np.sqrt(n), 5))

## ۴. خطای استاندارد Sharpe

In [ ]:
def sharpe(x):
    return x.mean() / x.std(ddof=1) * np.sqrt(365)

def sharpe_se(sr_annual, n_days):
    sr_daily = sr_annual / np.sqrt(365)
    return np.sqrt((1 + 0.5 * sr_daily**2) / n_days) * np.sqrt(365)

years = n / 365
sr = sharpe(r)
print(round(sr, 3), round(years, 2), round(sharpe_se(sr, n), 3), round(1 / np.sqrt(years), 3))

In [ ]:
rng = np.random.default_rng(7)
true_sr = 0.7
mu_d = true_sr / np.sqrt(365) * 0.03
normal = mu_d + 0.03 * rng.standard_normal(size=(4000, n))
fat = mu_d + 0.03 * rng.standard_t(4, size=(4000, n)) / np.sqrt(2)
for name, sim in [("normal", normal), ("t4", fat)]:
    est = sim.mean(axis=1) / sim.std(axis=1, ddof=1) * np.sqrt(365)
    print(name, round(est.mean(), 3), round(est.std(), 3))
print(round(sharpe_se(true_sr, n), 3))

## ۵. Bootstrap: خطا را بدون فرمول بساز

In [ ]:
rng = np.random.default_rng(8)
B = 5000
boot = r.values[rng.integers(0, n, size=(B, n))]
boot_mean = boot.mean(axis=1)
boot_sr = boot.mean(axis=1) / boot.std(axis=1, ddof=1) * np.sqrt(365)
print(np.percentile(boot_mean, [2.5, 97.5]).round(5), round(boot_mean.std(), 5))
print(np.percentile(boot_sr, [2.5, 97.5]).round(2), round(boot_sr.std(), 3))

## ۶. وقتی داده مستقل نیست: Block Bootstrap

In [ ]:
def block_bootstrap(x, block, B, rng):
    n = len(x)
    nb = int(np.ceil(n / block))
    starts = rng.integers(0, n - block + 1, size=(B, nb))
    idx = (starts[:, :, None] + np.arange(block)).reshape(B, -1)[:, :n]
    return x[idx]

rng = np.random.default_rng(9)
vol_boot = {}
for block in [1, 20, 60]:
    b = block_bootstrap((r.values - r.values.mean()) ** 2, block, 4000, rng)
    vol_boot[block] = b.mean(axis=1)
    print(block, round(np.sqrt(vol_boot[block]).std(), 5))
print(round(r.autocorr(1), 3), round((r ** 2).autocorr(1), 3))

## ۷. با پنج سال داده چه چیزی را می‌شود دید؟

In [ ]:
for target in [0.5, 1.0, 2.0]:
    need = 4 * (1 + 0.5 * (target / np.sqrt(365)) ** 2) / target**2
    print(target, round(need, 1))